# Chapter 48: Production Monitoring & Observability

**Volume 3 — Advanced Techniques for Production**

A router either forwards the packet or it doesn't. An LLM returns HTTP 200 whether the
answer is correct or completely hallucinated. **You cannot monitor AI systems like network gear.**

This chapter builds a complete observability stack from scratch — no external platforms needed.

### What You Will Build
1. **Instrumented LLM Client** — capture latency, tokens, cost, errors on every call (RED method)
2. **Sliding Window Metrics** — rolling p50/p95/p99 latency + QPS + error rate per time window
3. **LLM-as-Judge Quality Monitor** — automatically score faithfulness, relevance, hallucination
4. **Hallucination Detector** — self-consistency voting + Chain of Verification (CoVe)
5. **SLO Tracker + Cost Anomaly Detection** — error budget, burn rate, cost spike alerts

### The Core Problem AI Adds
```
Traditional monitoring:  HTTP 200 = success
AI monitoring:           HTTP 200 = "the request completed"
                                    (the answer may still be wrong)
```
Quality monitoring is the AI-specific observability challenge with no network equivalent.

In [ ]:
# pip install anthropic
# All monitoring built from pure Python standard library!

import os, json, time, math, hashlib, random
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Optional

# ── Anthropic client ───────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("Anthropic API connected")
else:
    _client = None
    print("No API key - simulation mode active (all monitoring still works!)")

MODEL = "claude-haiku-4-5-20251001"

# ── Realistic response simulator ───────────────────────────────────────────────
def _simulate(prompt: str, seed: int = None) -> tuple:
    'Returns (response_text, simulated_latency_ms, input_tokens, output_tokens).'
    if seed is not None:
        random.seed(seed)
    latency = random.uniform(400, 2800)
    p = prompt.lower()
    if "bgp" in p and "path" in p:
        text = ("BGP path selection order: LOCAL_PREF (highest wins), "
                "AS_PATH length (shortest wins), ORIGIN (IGP < EGP < ?), "
                "MED (lowest wins), eBGP over iBGP, lowest IGP metric to NEXT_HOP, "
                "lowest Router-ID.")
    elif "ospf" in p and "dr" in p:
        text = "OSPF DR election: highest interface priority wins (default 1), then highest Router-ID. DR never preempts once elected."
    elif "ospf" in p and "area" in p:
        text = "OSPF area types: Backbone (Area 0), Standard, Stub, Totally Stubby, NSSA. Stub = no external LSAs. Totally Stubby = no external + no summary LSAs."
    elif "qos" in p:
        text = "QoS DSCP: EF (46) for voice, AF41 for interactive video, CS3 for call signaling, BE (0) for best-effort."
    elif "bgp" in p and ("idle" in p or "session" in p):
        text = "BGP Idle causes: missing neighbor statement, MD5 mismatch, wrong update-source, ACL blocking TCP 179."
    elif "evaluate" in p or "score" in p or "judge" in p or "faithful" in p:
        # LLM-as-judge response
        score = random.randint(6, 10)
        text = json.dumps({
            "faithfulness": round(random.uniform(0.7, 1.0), 2),
            "relevance": round(random.uniform(0.75, 1.0), 2),
            "hallucination_risk": random.choice(["low", "low", "low", "medium"]),
            "score": score,
            "reasoning": f"Response addresses the question with score {score}/10"
        })
    elif "verify" in p or "verification" in p or "check" in p:
        text = random.choice([
            "Yes, BGP LOCAL_PREF is indeed used for path selection within an AS.",
            "Correct, OSPF elects DR on broadcast and NBMA segments.",
            "Yes, DSCP EF (46) is the standard marking for voice traffic."
        ])
    else:
        text = f"Network engineering response to: {prompt[:60]}..."

    in_tokens = len(prompt.split()) + 20
    out_tokens = len(text.split())
    return text, latency, in_tokens, out_tokens

print(f"Model: {MODEL}")
print("Monitoring stack will track: latency, tokens, cost, quality scores, error budget")

## Demo 1: Instrumented LLM Client — RED Method

The **RED Method** asks three questions about every service call:
- **R**ate — how many requests per second?
- **E**rrors — what fraction fail, and how?
- **D**uration — how long does each take?

For LLMs we also track: TTFT (Time To First Token), token counts, and cost per call.

Every call produces a **structured JSON log** — machine-parseable, queryable in Elasticsearch.

> **In Simple Words:** This is like SNMP polling on your router — every 30 seconds you collect
> interface counters (packets, errors, utilization). The instrumented LLM client does the same:
> every API call logs its metrics so you can trend them over time and alert on anomalies.

**Why structured logging matters:**
- Unstructured: `"LLM call completed in 1.2s"` — useless for debugging
- Structured: `{"latency_ms": 1200, "tokens_in": 450, "cost": 0.0023, "error": null}` — queryable

In [ ]:
# ── Demo 1: Instrumented LLM Client ──────────────────────────────────────────

@dataclass
class CallRecord:
    'One LLM API call - all metrics captured.'
    timestamp: float
    team: str
    prompt_hash: str       # SHA-256 prefix for privacy
    model: str
    tokens_input: int
    tokens_output: int
    latency_ms: float
    cost_usd: float
    error: Optional[str] = None
    tags: dict = field(default_factory=dict)

    def to_log_entry(self) -> dict:
        'Structured JSON log - ready for Elasticsearch/Splunk.'
        return {
            "timestamp": self.timestamp,
            "event": "llm_call",
            "team": self.team,
            "prompt_hash": self.prompt_hash,
            "model": self.model,
            "tokens_input": self.tokens_input,
            "tokens_output": self.tokens_output,
            "latency_ms": round(self.latency_ms, 1),
            "cost_usd": self.cost_usd,
            "error": self.error,
            "tags": self.tags,
        }


# Token pricing per 1M tokens (input_cost, output_cost)
PRICING = {
    "claude-sonnet-4-5-20250514":        (3.00, 15.00),
    "claude-haiku-4-5-20251001": (0.25,  1.25),
}


class InstrumentedClient:
    '''
    Wraps Anthropic API to capture full metrics on every call.

    Captures: latency, token counts, cost, error type, team identity.
    Emits:    structured JSON log per call.
    Tracks:   rolling error rate, total cost per team.
    '''

    def __init__(self, team: str = "default"):
        self.team = team
        self.history: list = []           # list of CallRecord
        self.team_costs = defaultdict(float)
        self._error_types = defaultdict(int)

    def _make_prompt_hash(self, prompt: str) -> str:
        return hashlib.md5(prompt.encode()).hexdigest()[:8]

    def chat(self, prompt: str, model: str = MODEL,
             max_tokens: int = 300, tags: dict = None) -> str:
        start = time.time()
        error = None
        tokens_in = tokens_out = 0
        result = ""

        try:
            if USE_API:
                resp = client.messages.create(
                    model=model, max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}]
                )
                result = resp.content[0].text
                tokens_in  = resp.usage.input_tokens
                tokens_out = resp.usage.output_tokens
            else:
                result, sim_latency, tokens_in, tokens_out = _simulate(prompt)
                # Simulate realistic latency
                time.sleep(sim_latency / 10000)   # scaled down for demo speed

        except Exception as e:
            error = type(e).__name__ + ": " + str(e)[:50]
            self._error_types[type(e).__name__] += 1

        latency_ms = (time.time() - start) * 1000
        price_in, price_out = PRICING.get(model, (3.00, 15.00))
        cost = (tokens_in * price_in + tokens_out * price_out) / 1_000_000

        record = CallRecord(
            timestamp=time.time(),
            team=self.team,
            prompt_hash=self._make_prompt_hash(prompt),
            model=model,
            tokens_input=tokens_in,
            tokens_output=tokens_out,
            latency_ms=latency_ms,
            cost_usd=round(cost, 6),
            error=error,
            tags=tags or {}
        )
        self.history.append(record)
        self.team_costs[self.team] += cost
        return result

    def red_report(self) -> dict:
        'RED method summary: Rate, Errors, Duration.'
        if not self.history:
            return {}
        total = len(self.history)
        errors = sum(1 for r in self.history if r.error)
        latencies = sorted(r.latency_ms for r in self.history)
        n = len(latencies)
        p50 = latencies[int(n * 0.50)]
        p95 = latencies[int(n * 0.95)]
        total_cost = sum(r.cost_usd for r in self.history)
        total_tokens = sum(r.tokens_input + r.tokens_output for r in self.history)

        return {
            "total_calls":    total,
            "error_count":    errors,
            "error_rate":     f"{errors/total*100:.1f}%",
            "p50_latency_ms": round(p50, 0),
            "p95_latency_ms": round(p95, 0),
            "total_cost_usd": round(total_cost, 4),
            "total_tokens":   total_tokens,
            "avg_cost_per_call": round(total_cost / total, 6),
        }

    def print_logs(self, last_n: int = 5):
        'Print last N structured log entries.'
        for record in self.history[-last_n:]:
            log = record.to_log_entry()
            status = "ERROR" if log["error"] else "OK"
            print(f"  [{status}] latency={log['latency_ms']:>7.1f}ms  "
                  f"tokens={log['tokens_input']}+{log['tokens_output']}  "
                  f"cost=${log['cost_usd']:.6f}  team={log['team']}")


# ── Demo: 3 teams making LLM calls ────────────────────────────────────────────
print("=== Instrumented LLM Client Demo ===")
print()

teams = {
    "NOC":      InstrumentedClient(team="NOC"),
    "Security": InstrumentedClient(team="Security"),
    "ChangeCtl":InstrumentedClient(team="ChangeCtl"),
}

noc_queries = [
    "What BGP attributes affect path selection?",
    "Explain OSPF DR election",
    "What are OSPF area types?",
    "QoS DSCP marking for voice",
    "BGP session stuck in Idle causes",
]
sec_queries = [
    "How to detect port scanning in network logs?",
    "What is a BGP hijack and how to detect it?",
]
chg_queries = [
    "Pre-change checklist for OSPF area migration",
    "Risk assessment for BGP policy change",
    "Post-change verification for QoS update",
]

for q in noc_queries:
    teams["NOC"].chat(q, tags={"endpoint": "bgp-analysis"})
for q in sec_queries:
    teams["Security"].chat(q, tags={"endpoint": "threat-analysis"})
for q in chg_queries:
    teams["ChangeCtl"].chat(q, tags={"endpoint": "change-mgmt"})

print("Structured log entries (last 5 from NOC team):")
teams["NOC"].print_logs(last_n=5)

print()
print("RED Method Summary by Team:")
print(f"{'Team':<12} {'Calls':>6} {'Errors':>7} {'p50(ms)':>9} {'p95(ms)':>9} "
      f"{'Total $':>9} {'$/call':>9}")
print("-" * 65)
for team_name, client in teams.items():
    r = client.red_report()
    print(f"{team_name:<12} {r['total_calls']:>6} {r['error_count']:>7} "
          f"{r['p50_latency_ms']:>9.0f} {r['p95_latency_ms']:>9.0f} "
          f"${r['total_cost_usd']:>8.4f} ${r['avg_cost_per_call']:>8.6f}")

## Demo 2: Sliding Window Metrics — Rolling p95 + QPS

A simple average hides what matters. If 99% of requests take 200ms but 1% take 30 seconds,
your average is 500ms — fine! But users of that 1% experience a broken product.

**Percentile metrics tell the truth:**
- p50 (median) = typical experience
- p95 = worst experience for 1 in 20 users
- p99 = worst experience for 1 in 100 users

**Window types:**
- **Tumbling window**: fixed 60-second blocks (non-overlapping)
- **Hopping window**: 5-minute window that moves every 1 minute (smooth trends)

> **In Simple Words:** This is exactly how RMON (Remote Monitoring) works on network gear.
> RMON2 uses sliding time windows to compute interface utilization averages and thresholds.
> A 5-minute hopping window smooths out micro-bursts that would alarm on a 10-second tumbling window.

In [ ]:
# ── Demo 2: Sliding Window Metrics ───────────────────────────────────────────

class MetricsWindow:
    '''
    Sliding window metrics collector.

    Keeps a deque of (timestamp, value) tuples.
    Automatically drops entries older than window_seconds on each read.
    '''
    def __init__(self, window_seconds: int = 300, name: str = "metric"):
        self.window_seconds = window_seconds
        self.name = name
        self._data = deque()   # (timestamp, value)

    def record(self, value: float):
        self._data.append((time.time(), value))

    def _prune(self):
        cutoff = time.time() - self.window_seconds
        while self._data and self._data[0][0] < cutoff:
            self._data.popleft()

    def values(self) -> list:
        self._prune()
        return [v for _, v in self._data]

    def percentile(self, p: float) -> float:
        'p in range 0-100. Returns the p-th percentile of current window.'
        vals = sorted(self.values())
        if not vals:
            return 0.0
        idx = int(len(vals) * p / 100)
        idx = min(idx, len(vals) - 1)
        return vals[idx]

    def mean(self) -> float:
        vals = self.values()
        return sum(vals) / len(vals) if vals else 0.0

    def count(self) -> int:
        self._prune()
        return len(self._data)

    def rate_per_second(self) -> float:
        return self.count() / self.window_seconds


class ServiceDashboard:
    '''
    Real-time dashboard for a single AI service endpoint.
    Tracks latency, throughput (QPS), error rate, and token usage
    using separate sliding windows.
    '''
    def __init__(self, service_name: str, window_sec: int = 300):
        self.service_name = service_name
        self.latency   = MetricsWindow(window_sec, "latency_ms")
        self.errors    = MetricsWindow(window_sec, "errors")    # 1=error, 0=ok
        self.tokens_in = MetricsWindow(window_sec, "tokens_in")
        self.costs     = MetricsWindow(window_sec, "cost_usd")
        self._total_calls = 0

    def record_call(self, latency_ms: float, tokens_in: int,
                    cost_usd: float, is_error: bool = False):
        self.latency.record(latency_ms)
        self.errors.record(1.0 if is_error else 0.0)
        self.tokens_in.record(float(tokens_in))
        self.costs.record(cost_usd)
        self._total_calls += 1

    def snapshot(self) -> dict:
        error_vals = self.errors.values()
        error_rate = (sum(error_vals) / len(error_vals) * 100) if error_vals else 0.0
        return {
            "service":        self.service_name,
            "qps":            round(self.latency.rate_per_second(), 3),
            "p50_ms":         round(self.latency.percentile(50), 0),
            "p95_ms":         round(self.latency.percentile(95), 0),
            "p99_ms":         round(self.latency.percentile(99), 0),
            "error_rate_pct": round(error_rate, 2),
            "cost_per_min":   round(sum(self.costs.values()) / (self.costs.window_seconds / 60), 4),
            "window_calls":   self.latency.count(),
        }


# ── Simulate 60 seconds of traffic with realistic patterns ────────────────────
random.seed(42)

dash = ServiceDashboard("bgp-analysis-api", window_sec=60)

# Simulate traffic: mostly fast, some slow, a few errors
print("=== Simulating 80 API calls with realistic latency distribution ===")

call_scenarios = []
for i in range(80):
    is_error = (i % 20 == 0)          # 5% error rate
    is_slow  = (i % 10 == 0)          # 10% slow calls (network congestion)

    if is_error:
        latency = random.uniform(5000, 10000)   # timeout
        tokens_in = 0
        cost = 0.0
    elif is_slow:
        latency = random.uniform(3000, 6000)    # slow LLM call
        tokens_in = random.randint(300, 600)
        cost = tokens_in * 0.25 / 1_000_000
    else:
        latency = random.uniform(200, 800)       # normal
        tokens_in = random.randint(100, 400)
        cost = tokens_in * 0.25 / 1_000_000

    call_scenarios.append((latency, tokens_in, cost, is_error))
    dash.record_call(latency, tokens_in, cost, is_error)

snap = dash.snapshot()
print()
print("=== 60-Second Rolling Window Snapshot ===")
print(f"  Service:        {snap['service']}")
print(f"  QPS:            {snap['qps']:.3f} req/s")
print(f"  p50 latency:    {snap['p50_ms']:.0f} ms  (typical user experience)")
print(f"  p95 latency:    {snap['p95_ms']:.0f} ms  (1 in 20 users experience this)")
print(f"  p99 latency:    {snap['p99_ms']:.0f} ms  (1 in 100 users experience this)")
print(f"  Error rate:     {snap['error_rate_pct']:.1f}%")
print(f"  Cost/minute:    ${snap['cost_per_min']:.4f}")
print(f"  Window calls:   {snap['window_calls']}")

# ── Show why average is misleading ────────────────────────────────────────────
all_latencies = [s[0] for s in call_scenarios]
mean_lat = sum(all_latencies) / len(all_latencies)
sorted_lat = sorted(all_latencies)
p95_lat = sorted_lat[int(len(sorted_lat) * 0.95)]
p99_lat = sorted_lat[int(len(sorted_lat) * 0.99)]

print()
print("=== Why Average is Misleading ===")
print(f"  Average latency:  {mean_lat:.0f} ms  <- hides the pain")
print(f"  p95 latency:      {p95_lat:.0f} ms  <- 4 users per 80 experience this")
print(f"  p99 latency:      {p99_lat:.0f} ms  <- 1 user per 80 experiences this")
print()
print("  SLO: p95 < 3000ms")
if p95_lat < 3000:
    print(f"  Status: PASSING (p95={p95_lat:.0f}ms < 3000ms)")
else:
    print(f"  Status: BREACHING (p95={p95_lat:.0f}ms > 3000ms) - alert!")

# ── Tumbling vs Hopping windows ───────────────────────────────────────────────
print()
print("=== Tumbling vs Hopping Window Comparison ===")

# Tumbling: non-overlapping 20-call blocks
block_size = 20
print("Tumbling windows (non-overlapping, 20 calls each):")
for block_start in range(0, 80, block_size):
    block = all_latencies[block_start:block_start + block_size]
    block_p95 = sorted(block)[int(len(block) * 0.95)]
    print(f"  Window {block_start//block_size + 1} (calls {block_start+1}-{block_start+block_size}): "
          f"p95={block_p95:.0f}ms, avg={sum(block)/len(block):.0f}ms")

# Hopping: overlapping windows (20 calls, step=10)
print("Hopping windows (overlapping, 20 calls, step=10):")
for start in range(0, 70, 10):
    block = all_latencies[start:start + 20]
    if len(block) < 20:
        break
    block_p95 = sorted(block)[int(len(block) * 0.95)]
    print(f"  Window [{start+1}-{start+20}]: p95={block_p95:.0f}ms (smoother trend)")

## Demo 3: LLM-as-Judge Quality Monitor

HTTP 200 tells you the request completed. It does NOT tell you if the answer is correct.

**LLM-as-Judge**: use a powerful LLM to score your AI system's responses against:
- **Faithfulness** — are claims supported by the context? (0.0 to 1.0)
- **Relevance** — does the response address the actual question? (0.0 to 1.0)
- **Hallucination Risk** — did the model invent things not in the context? (low/medium/high)

Run on a **5-10% sample** of production traffic. Track scores over time.
A sudden drop in faithfulness = your RAG retrieval just broke.

> **In Simple Words:** LLM-as-Judge is like running `show ip bgp` before and after a
> policy change. You are not just checking if the command ran (HTTP 200) — you are
> checking if the routing table looks correct (quality of the output).

**Three important limitations to know:**
1. Probabilistic — same response may score differently on different runs
2. Position bias — judges prefer whichever answer appears first in a comparison
3. Self-bias — a model tends to rate its own outputs higher

In [ ]:
# ── Demo 3: LLM-as-Judge Quality Monitor ─────────────────────────────────────

def llm_call_for_judge(prompt: str, system: str = "") -> str:
    'LLM call for evaluation purposes.'
    if USE_API:
        kwargs = {
            "model": MODEL, "max_tokens": 400,
            "messages": [{"role": "user", "content": prompt}]
        }
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    else:
        result, _, _, _ = _simulate(prompt, seed=hash(prompt) % 1000)
        # Return a valid JSON evaluation for simulation
        return json.dumps({
            "faithfulness": round(random.uniform(0.65, 0.98), 2),
            "relevance": round(random.uniform(0.70, 0.99), 2),
            "hallucination_risk": random.choice(["low", "low", "low", "medium", "high"]),
            "score": random.randint(6, 10),
            "reasoning": "Response addresses the question with appropriate technical detail."
        })


def judge_response(question: str, context: str, response: str) -> dict:
    '''
    Use LLM-as-Judge to evaluate a response.
    Returns scores for faithfulness, relevance, hallucination risk.

    Judge prompt design principles:
    1. Specific rubric (not "is this good?" but scored criteria)
    2. Temperature=0 for determinism
    3. JSON output for parseable scores
    '''
    system_prompt = '''You are an expert network engineering evaluator.
Evaluate the AI response against the provided context and question.

Score STRICTLY based on:
- faithfulness (0.0-1.0): fraction of claims supported by the context
- relevance (0.0-1.0): how well the response addresses the specific question
- hallucination_risk: "low", "medium", or "high"
- score (1-10): overall quality

Return ONLY valid JSON: {"faithfulness": float, "relevance": float,
"hallucination_risk": string, "score": int, "reasoning": string}'''

    judge_prompt = f'''QUESTION: {question}

CONTEXT PROVIDED TO AI:
{context}

AI RESPONSE TO EVALUATE:
{response}

Evaluate the response. Return JSON only.'''

    raw = llm_call_for_judge(judge_prompt, system=system_prompt)
    try:
        # Extract JSON from response
        start = raw.find('{')
        end = raw.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(raw[start:end])
    except (json.JSONDecodeError, ValueError):
        pass
    # Fallback
    return {"faithfulness": 0.5, "relevance": 0.5,
            "hallucination_risk": "unknown", "score": 5,
            "reasoning": "Parse error in judge output"}


class QualityMonitor:
    '''
    Tracks LLM response quality over time using LLM-as-Judge.

    Maintains rolling history of quality scores.
    Alerts when scores drop below thresholds.
    '''
    def __init__(self, faithfulness_threshold: float = 0.80,
                 relevance_threshold: float = 0.75):
        self.faithfulness_threshold = faithfulness_threshold
        self.relevance_threshold = relevance_threshold
        self._scores = []   # list of evaluation dicts

    def evaluate(self, question: str, context: str,
                 response: str, source_id: str = "") -> dict:
        scores = judge_response(question, context, response)
        scores["source_id"] = source_id
        scores["timestamp"] = time.time()
        self._scores.append(scores)

        # Check thresholds
        alerts = []
        if scores.get("faithfulness", 1.0) < self.faithfulness_threshold:
            alerts.append(f"LOW FAITHFULNESS: {scores['faithfulness']:.2f} < {self.faithfulness_threshold}")
        if scores.get("relevance", 1.0) < self.relevance_threshold:
            alerts.append(f"LOW RELEVANCE: {scores['relevance']:.2f} < {self.relevance_threshold}")
        if scores.get("hallucination_risk") == "high":
            alerts.append("HIGH HALLUCINATION RISK detected")
        scores["alerts"] = alerts
        return scores

    def summary(self) -> dict:
        if not self._scores:
            return {}
        faith_vals = [s.get("faithfulness", 0) for s in self._scores]
        rel_vals   = [s.get("relevance", 0) for s in self._scores]
        risk_counts = defaultdict(int)
        for s in self._scores:
            risk_counts[s.get("hallucination_risk", "unknown")] += 1
        return {
            "evaluations": len(self._scores),
            "avg_faithfulness": round(sum(faith_vals) / len(faith_vals), 3),
            "avg_relevance":    round(sum(rel_vals) / len(rel_vals), 3),
            "min_faithfulness": round(min(faith_vals), 3),
            "min_relevance":    round(min(rel_vals), 3),
            "hallucination_risk_distribution": dict(risk_counts),
            "alerts_fired": sum(1 for s in self._scores if s.get("alerts")),
        }


# ── Test production responses ──────────────────────────────────────────────────
BGP_CONTEXT = '''BGP path selection uses these attributes in order:
1. Highest LOCAL_PREF (100 default)
2. Shortest AS_PATH
3. Lowest ORIGIN (IGP < EGP < ?)
4. Lowest MED (between same AS paths)
5. eBGP preferred over iBGP
6. Lowest IGP metric to NEXT_HOP
7. Oldest eBGP path (stability)
8. Lowest Router-ID as tiebreaker'''

test_cases = [
    {
        "id": "good-response",
        "question": "How does BGP select the best path?",
        "context": BGP_CONTEXT,
        "response": ("BGP selects the best path using LOCAL_PREF first (highest wins), "
                     "then shortest AS_PATH, then ORIGIN code (IGP preferred), "
                     "then MED (lowest wins for same AS), then eBGP over iBGP, "
                     "then lowest IGP metric to NEXT_HOP, and finally Router-ID as tiebreaker.")
    },
    {
        "id": "partially-hallucinated",
        "question": "How does BGP select the best path?",
        "context": BGP_CONTEXT,
        "response": ("BGP selects the best path primarily based on LOCAL_PREF. "
                     "It also considers BGP community values and route reflector cluster IDs. "
                     "The WEIGHT attribute (proprietary) is checked before LOCAL_PREF on Cisco. "
                     "Shortest AS_PATH and lowest MED are also used.")
        # Note: WEIGHT is Cisco-specific and not in context; communities/RR-clusters not mentioned
    },
    {
        "id": "irrelevant-response",
        "question": "How does BGP select the best path?",
        "context": BGP_CONTEXT,
        "response": ("OSPF is a link-state routing protocol that uses Dijkstra's algorithm "
                     "to find the shortest path. It uses areas to scale, with Area 0 as backbone. "
                     "Hello packets are sent every 10 seconds by default.")
        # Completely wrong topic
    },
]

qmonitor = QualityMonitor(faithfulness_threshold=0.80, relevance_threshold=0.75)

print("=== LLM-as-Judge Quality Evaluation ===")
print()

for tc in test_cases:
    print(f"Evaluating: {tc['id']}")
    result = qmonitor.evaluate(
        question=tc["question"],
        context=tc["context"],
        response=tc["response"],
        source_id=tc["id"]
    )
    faith_icon = "OK" if result.get("faithfulness", 0) >= 0.80 else "ALERT"
    rel_icon   = "OK" if result.get("relevance", 0) >= 0.75 else "ALERT"
    print(f"  Faithfulness:      {result.get('faithfulness', '?'):.2f}  [{faith_icon}]")
    print(f"  Relevance:         {result.get('relevance', '?'):.2f}  [{rel_icon}]")
    print(f"  Hallucination:     {result.get('hallucination_risk', '?')}")
    print(f"  Overall score:     {result.get('score', '?')}/10")
    print(f"  Reasoning:         {result.get('reasoning', '')[:80]}")
    if result.get("alerts"):
        for alert in result["alerts"]:
            print(f"  *** ALERT: {alert}")
    print()

print("=== Quality Monitor Summary ===")
summary = qmonitor.summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

## Demo 4: Hallucination Detection — Self-Consistency + Chain of Verification

Two techniques to catch hallucinations before they reach users:

**1. Self-Consistency Voting**
Run the same query N times. Count how often each answer agrees.
- High agreement (4/5 same) = confident answer
- Low agreement (2/5 different) = model is uncertain -> flag for human review

**2. Chain of Verification (CoVe)**
1. Generate the answer
2. Generate verification questions about the answer's specific claims
3. Answer each verification question independently
4. Compare: does the verification contradict the original claim?

> **In Simple Words:** Self-consistency is like getting 5 BGP routers to calculate the best
> path independently. If all 5 choose the same next-hop, you trust it. If 3 go left and 2 go
> right, there is a tie-breaking problem — route to human review before pushing the config.

**When to use:**
- Flag responses for human review (never auto-apply flagged configs)
- Tune the consistency threshold based on domain risk

In [ ]:
# ── Demo 4: Hallucination Detection ──────────────────────────────────────────

def multi_call(prompt: str, n: int = 3) -> list:
    'Call LLM N times and return all responses (for consistency checking).'
    responses = []
    for i in range(n):
        if USE_API:
            resp = client.messages.create(
                model=MODEL, max_tokens=200,
                messages=[{"role": "user", "content": prompt}]
            )
            responses.append(resp.content[0].text.strip())
        else:
            # Simulate with slight variation
            base, _, _, _ = _simulate(prompt, seed=i * 42)
            # Add occasional variation to simulate model uncertainty
            if i == 2 and random.random() < 0.3:
                base = base.replace("LOCAL_PREF", "weight").replace("AS_PATH", "community")
            responses.append(base)
    return responses


def self_consistency_check(prompt: str, n_samples: int = 3) -> dict:
    '''
    Self-consistency: run query N times, measure agreement.

    Agreement measured by longest-common-subsequence of key terms.
    In production: use embedding cosine similarity for semantic agreement.
    '''
    responses = multi_call(prompt, n=n_samples)

    # Extract key noun phrases (simplified - in production use NER or embeddings)
    def key_terms(text: str) -> set:
        import re
        # Extract capitalized words and technical terms
        terms = set(re.findall(r'[A-Z][A-Z_-]+|[a-z]+_[a-z]+', text))
        # Also grab numbers and specific protocol terms
        terms.update(re.findall(r'(?:BGP|OSPF|QoS|DSCP|EF|AF|CS|iBGP|eBGP|MED)', text))
        return terms

    term_sets = [key_terms(r) for r in responses]

    # Measure pairwise Jaccard similarity
    pairwise_sim = []
    for i in range(len(term_sets)):
        for j in range(i + 1, len(term_sets)):
            a, b = term_sets[i], term_sets[j]
            union = a | b
            inter = a & b
            sim = len(inter) / len(union) if union else 0.0
            pairwise_sim.append(sim)

    avg_agreement = sum(pairwise_sim) / len(pairwise_sim) if pairwise_sim else 0.0

    confidence = "HIGH" if avg_agreement >= 0.70 else (
                 "MEDIUM" if avg_agreement >= 0.50 else "LOW")

    return {
        "n_samples":      n_samples,
        "avg_agreement":  round(avg_agreement, 3),
        "confidence":     confidence,
        "flag_for_review": avg_agreement < 0.60,
        "responses":      responses,
    }


def chain_of_verification(original_answer: str, topic: str) -> dict:
    '''
    Chain of Verification (CoVe):
    1. Generate verification questions for key claims in the answer
    2. Answer each question independently
    3. Check if verification contradicts the original claim

    Simplified implementation - in production use structured LLM calls.
    '''
    # Step 1: Generate verification questions (simulated extraction)
    verification_questions = []

    # Parse key claims from the answer (simplified rule-based extraction)
    claim_patterns = [
        ("LOCAL_PREF", "Is LOCAL_PREF the FIRST attribute checked in BGP path selection?"),
        ("AS_PATH",    "Is AS_PATH length used in BGP path selection (shorter = better)?"),
        ("MED",        "Is MED used to prefer paths from the same AS?"),
        ("iBGP",       "Is eBGP preferred over iBGP in BGP path selection?"),
        ("Router-ID",  "Is Router-ID used as a tiebreaker in BGP path selection?"),
        ("OSPF",       "Does OSPF selection use BGP path attributes like LOCAL_PREF?"),
        ("weight",     "Is WEIGHT a standard BGP attribute defined in RFC?"),
        ("community",  "Does BGP community affect the core path selection order?"),
    ]

    active_claims = [(term, q) for term, q in claim_patterns if term.lower() in original_answer.lower()]

    if not active_claims:
        return {"verified": True, "contradictions": [], "confidence": "HIGH"}

    # Step 2: Answer each verification question
    contradictions = []
    for term, question in active_claims:
        if USE_API:
            resp = client.messages.create(
                model=MODEL, max_tokens=100,
                system="Answer yes or no, then one sentence explanation. Be precise about networking facts.",
                messages=[{"role": "user", "content": question}]
            )
            verification_answer = resp.content[0].text.strip().lower()
        else:
            # Simulate verification - most technical claims check out
            if term in ["LOCAL_PREF", "AS_PATH", "MED", "iBGP", "Router-ID"]:
                verification_answer = "yes, this is correct standard BGP behavior."
            elif term in ["OSPF", "weight", "community"]:
                verification_answer = "no, this is not accurate for standard BGP path selection."
            else:
                verification_answer = "yes, confirmed."

        # Step 3: Check for contradictions
        is_contradiction = (
            ("no" in verification_answer[:10] or "incorrect" in verification_answer[:30]) and
            term in original_answer
        )
        if is_contradiction:
            contradictions.append({
                "claim": f"Answer mentions '{term}'",
                "verification_question": question,
                "verification_answer": verification_answer[:100],
                "verdict": "POSSIBLE HALLUCINATION"
            })

    return {
        "claims_checked": len(active_claims),
        "contradictions": contradictions,
        "verified": len(contradictions) == 0,
        "confidence": "LOW" if contradictions else "HIGH",
    }


# ── Test on good vs. hallucinated responses ───────────────────────────────────
print("=== Self-Consistency Check ===")
print()

test_prompts = [
    ("High-confidence query (clear facts)", "What BGP attributes affect path selection?"),
    ("Medium-confidence (more nuanced)",    "What BGP attributes affect path selection in NSSA areas?"),
]

for label, prompt in test_prompts:
    print(f"Query: {label}")
    result = self_consistency_check(prompt, n_samples=3)
    flag_icon = "FLAG FOR REVIEW" if result["flag_for_review"] else "AUTO-SERVE OK"
    print(f"  Samples:         {result['n_samples']}")
    print(f"  Avg agreement:   {result['avg_agreement']:.3f}")
    print(f"  Confidence:      {result['confidence']}")
    print(f"  Decision:        {flag_icon}")
    print()

print("=== Chain of Verification (CoVe) ===")
print()

answers_to_check = [
    {
        "label":  "Correct answer",
        "answer": ("BGP path selection: LOCAL_PREF (highest wins), AS_PATH (shortest wins), "
                   "ORIGIN, MED (lowest wins), eBGP over iBGP, IGP metric, Router-ID tiebreaker.")
    },
    {
        "label":  "Hallucinated answer (contains wrong claims)",
        "answer": ("BGP path selection: weight (Cisco-specific, highest wins), LOCAL_PREF, "
                   "community values, AS_PATH. OSPF costs also affect BGP decisions. "
                   "community attribute determines the core path selection order.")
    },
]

for test in answers_to_check:
    print(f"Checking: {test['label']}")
    print(f"  Answer: {test['answer'][:90]}...")
    result = chain_of_verification(test["answer"], topic="BGP")
    print(f"  Claims checked:    {result['claims_checked']}")
    print(f"  Contradictions:    {len(result['contradictions'])}")
    print(f"  Verified:          {result['verified']}")
    print(f"  Confidence:        {result['confidence']}")
    if result["contradictions"]:
        for c in result["contradictions"]:
            print(f"  !! {c['verdict']}: {c['claim']}")
            print(f"     Verification Q: {c['verification_question']}")
    print()

## Demo 5: SLO Tracker + Cost Anomaly Detection

**SLO (Service Level Objective)** defines what "good" looks like — measurable, time-bounded.

**Error Budget** = the allowed failures before you breach the SLO.
**Burn Rate** = how fast you are consuming that budget.

```
SLO: 99% success rate over 30 days
Error budget: 1% of requests = 14.4 min downtime/day
Burn rate 1.0 = using budget at exactly the replenishment rate (fine)
Burn rate 10.0 = budget exhausted in 3 days (alert!)
```

**Cost Anomaly Detection** uses z-score to flag spikes:
- Compute rolling mean + standard deviation of cost per minute
- Alert when current cost > mean + 3*std (>3 sigma = anomaly)

> **In Simple Words:**
> SLO + error budget = routing policy with route dampening.
> Normal flapping = within dampening threshold (burn rate < 2.0).
> Excessive flapping = dampening kicks in (burn rate > 3.0 = page the team).
> Burn rate alerting = OSPF flood rate limiting trigger.

In [ ]:
# ── Demo 5: SLO Tracker + Cost Anomaly Detection ─────────────────────────────

@dataclass
class SLO:
    'Service Level Objective definition.'
    name: str
    metric: str             # e.g., "success_rate", "p95_latency_ms"
    target: float           # e.g., 0.99 for 99%
    window_days: int = 30
    comparison: str = ">="  # ">=" means metric must be >= target

    @property
    def error_budget_pct(self) -> float:
        if self.comparison == ">=":
            return (1.0 - self.target) * 100
        else:
            return self.target * 0.05   # 5% headroom above target

    def is_passing(self, current_value: float) -> bool:
        if self.comparison == ">=":
            return current_value >= self.target
        else:   # "<="
            return current_value <= self.target


class SLOTracker:
    '''
    Tracks error budget consumption and burn rate for a set of SLOs.

    Error budget = total allowed failures in the time window.
    Burn rate = current consumption rate / steady-state consumption rate.
    '''
    def __init__(self, slos: list):
        self.slos = {s.name: s for s in slos}
        self._events = deque()    # (timestamp, is_success, latency_ms, cost_usd)
        self._window_seconds = 30 * 24 * 3600   # 30 days

    def record(self, is_success: bool, latency_ms: float = 0.0, cost_usd: float = 0.0):
        self._events.append((time.time(), is_success, latency_ms, cost_usd))
        # Prune old events
        cutoff = time.time() - self._window_seconds
        while self._events and self._events[0][0] < cutoff:
            self._events.popleft()

    def _current_metrics(self) -> dict:
        events = list(self._events)
        if not events:
            return {"success_rate": 1.0, "error_rate": 0.0, "p95_latency_ms": 0.0}
        total = len(events)
        successes = sum(1 for _, ok, _, _ in events if ok)
        latencies = sorted(e[2] for e in events if e[2] > 0)
        p95 = latencies[int(len(latencies) * 0.95)] if latencies else 0.0
        return {
            "total_requests":   total,
            "success_rate":     round(successes / total, 4),
            "error_rate":       round((total - successes) / total, 4),
            "p95_latency_ms":   round(p95, 0),
        }

    def burn_rate(self, slo_name: str, short_window_sec: int = 3600) -> float:
        '''
        Burn rate = how fast error budget is being consumed.
        burn_rate 1.0 = exactly consuming at steady-state rate.
        burn_rate 10.0 = will exhaust 30-day budget in 3 days.
        '''
        slo = self.slos.get(slo_name)
        if not slo or not self._events:
            return 0.0
        cutoff = time.time() - short_window_sec
        recent = [e for e in self._events if e[0] >= cutoff]
        if not recent:
            return 0.0
        total = len(recent)
        if slo.metric == "success_rate":
            successes = sum(1 for _, ok, _, _ in recent if ok)
            short_error_rate = (total - successes) / total
            expected_error_rate = 1.0 - slo.target   # e.g., 0.01 for 99% SLO
            if expected_error_rate == 0:
                return float('inf') if short_error_rate > 0 else 0.0
            return short_error_rate / expected_error_rate
        return 0.0

    def status_report(self) -> dict:
        metrics = self._current_metrics()
        report = {"metrics": metrics, "slo_status": {}}
        for name, slo in self.slos.items():
            current_val = metrics.get(slo.metric, 0.0)
            passing = slo.is_passing(current_val)
            br = self.burn_rate(name)
            report["slo_status"][name] = {
                "target":       slo.target,
                "current":      current_val,
                "passing":      passing,
                "burn_rate":    round(br, 2),
                "alert":        br > 2.0,
                "error_budget_pct": slo.error_budget_pct,
            }
        return report


class CostAnomalyDetector:
    '''
    Detects unusual spikes in LLM token costs using rolling z-score.

    Baseline: rolling 20-sample mean and standard deviation.
    Alert: cost > mean + 3*std (3-sigma rule).

    Common causes of cost spikes:
    - Agent stuck in reasoning loop (keeps calling LLM)
    - Prompt injection inflating context size
    - Monitoring job running too frequently
    - Model upgrade with higher verbosity
    '''
    def __init__(self, window_size: int = 20, threshold_sigma: float = 3.0):
        self.window_size = window_size
        self.threshold_sigma = threshold_sigma
        self._cost_history = deque(maxlen=window_size)
        self.anomalies = []

    def record(self, cost_usd: float, context: str = "") -> dict:
        self._cost_history.append(cost_usd)
        if len(self._cost_history) < 5:
            return {"is_anomaly": False, "reason": "insufficient history"}

        costs = list(self._cost_history)
        mean = sum(costs) / len(costs)
        variance = sum((c - mean) ** 2 for c in costs) / len(costs)
        std = math.sqrt(variance) if variance > 0 else 0.001
        z_score = (cost_usd - mean) / std

        is_anomaly = z_score > self.threshold_sigma
        result = {
            "cost_usd":   round(cost_usd, 6),
            "mean":       round(mean, 6),
            "std":        round(std, 6),
            "z_score":    round(z_score, 2),
            "is_anomaly": is_anomaly,
            "context":    context,
        }
        if is_anomaly:
            self.anomalies.append(result)
        return result


# ── Run full SLO + Cost Anomaly demo ─────────────────────────────────────────
random.seed(99)

slos = [
    SLO("availability",   "success_rate",   target=0.99),
    SLO("latency_p95",    "p95_latency_ms", target=3000.0, comparison="<="),
]
tracker = SLOTracker(slos)
cost_detector = CostAnomalyDetector(window_size=15, threshold_sigma=3.0)

# Simulate 100 normal requests + 1 cost spike event
print("=== Simulating 100 API calls (normal traffic) ===")
for i in range(100):
    is_error = (random.random() < 0.005)    # 0.5% error rate (well within 1% SLO)
    latency = random.uniform(200, 2500) if not is_error else random.uniform(5000, 10000)
    cost = random.uniform(0.0001, 0.002)
    tracker.record(not is_error, latency_ms=latency, cost_usd=cost)
    cost_detector.record(cost, context=f"normal-call-{i}")

print("=== SLO Status (after 100 normal calls) ===")
report = tracker.status_report()
metrics = report["metrics"]
print(f"  Total requests:  {metrics['total_requests']}")
print(f"  Success rate:    {metrics['success_rate']*100:.2f}%")
print(f"  p95 latency:     {metrics['p95_latency_ms']:.0f} ms")
print()
print(f"{'SLO Name':<20} {'Target':>8} {'Current':>9} {'Passing':>9} {'Burn Rate':>10} {'Alert':>7}")
print("-" * 65)
for name, status in report["slo_status"].items():
    passing = "YES" if status["passing"] else "NO"
    alert   = "ALERT" if status["alert"] else "-"
    curr    = f"{status['current']*100:.2f}%" if "rate" in name else f"{status['current']:.0f}ms"
    tgt     = f"{status['target']*100:.1f}%" if "rate" in name else f"{status['target']:.0f}ms"
    print(f"{name:<20} {tgt:>8} {curr:>9} {passing:>9} {status['burn_rate']:>10.2f} {alert:>7}")

# ── Simulate a bad period ─────────────────────────────────────────────────────
print()
print("=== Simulating degradation event (elevated errors for 10 calls) ===")
for i in range(10):
    # 25% error rate (far above 1% SLO)
    is_error = (random.random() < 0.25)
    latency = random.uniform(200, 4000) if not is_error else random.uniform(8000, 15000)
    tracker.record(not is_error, latency_ms=latency)
    cost_detector.record(random.uniform(0.0001, 0.002))

report2 = tracker.status_report()
print(f"After degradation - Success rate: {report2['metrics']['success_rate']*100:.2f}%")
print()
print(f"{'SLO Name':<20} {'Passing':>9} {'Burn Rate':>10} {'Alert':>7}")
print("-" * 45)
for name, status in report2["slo_status"].items():
    passing = "YES" if status["passing"] else "BREACHING!"
    alert   = "!!! PAGE ON-CALL" if status["alert"] else "-"
    print(f"{name:<20} {passing:>9} {status['burn_rate']:>10.2f} {alert}")

# ── Cost anomaly detection ────────────────────────────────────────────────────
print()
print("=== Cost Anomaly Detection ===")
print("Simulating normal calls then sudden spike (agent in reasoning loop)...")

# Normal baseline
for i in range(15):
    cost_detector.record(random.uniform(0.0001, 0.0005), context="normal")

# Simulate a runaway agent (10x normal cost)
spike_scenarios = [
    (0.0003, "normal"),
    (0.0004, "normal"),
    (0.0042, "agent-loop-detected"),   # 10x spike
    (0.0038, "agent-loop-detected"),   # still high
    (0.0003, "normal"),
]

print(f"{'Call':<5} {'Cost':>10} {'z-score':>9} {'Anomaly':>8} {'Context'}")
print("-" * 55)
for i, (cost, ctx) in enumerate(spike_scenarios, 1):
    result = cost_detector.record(cost, context=ctx)
    anomaly = "*** ALERT" if result["is_anomaly"] else "-"
    print(f"{i:<5} ${cost:<9.4f} {result['z_score']:>9.2f} {anomaly:>8}  {ctx}")

print()
print(f"Total anomalies detected: {len(cost_detector.anomalies)}")
if cost_detector.anomalies:
    print("Anomaly details:")
    for a in cost_detector.anomalies:
        print(f"  Cost=${a['cost_usd']:.4f}, z-score={a['z_score']:.2f}, context={a['context']}")
        print(f"  Baseline mean=${a['mean']:.4f}, std=${a['std']:.4f}")

## Summary — What You Built

| Demo | Tool | What It Catches |
|------|------|-----------------|
| 1 | Instrumented Client | Latency spikes, cost per team, error types (RED method) |
| 2 | Sliding Window Metrics | Real p95/p99 percentiles — not misleading averages |
| 3 | LLM-as-Judge | Silent quality degradation (HTTP 200 but wrong answer) |
| 4 | Hallucination Detector | Inconsistent or factually wrong AI responses |
| 5 | SLO + Cost Anomaly | Budget breaches, burn rate spikes, runaway agents |

### The Four Monitoring Layers
```
Infrastructure layer  (USE method):   CPU, GPU, memory, queue depth
Service layer         (RED method):   QPS, error rate, TTFT + TPOT latency
Quality layer         (AI-specific):  faithfulness, relevance, hallucination rate
Cost layer            (AI-specific):  tokens/team/hour, z-score anomaly detection
```

### Production Alert Thresholds (starting point, tune to your traffic)
| Metric | Green | Yellow | Red |
|--------|-------|--------|-----|
| p95 latency | < 3s | 3-5s | > 5s |
| Error rate | < 0.5% | 0.5-2% | > 2% |
| Faithfulness | > 0.85 | 0.75-0.85 | < 0.75 |
| SLO burn rate | < 1.5 | 1.5-3.0 | > 3.0 |
| Cost z-score | < 2.0 | 2.0-3.0 | > 3.0 |

### Next Chapter
**Chapter 51 — Scaling AI Systems**: what to do when monitoring tells you that you are
hitting capacity limits — horizontal scaling, load balancing across model replicas,
autoscaling based on queue depth.

In [ ]:
# ── Integration: Full Production Monitoring Stack ─────────────────────────────

print("=" * 65)
print("INTEGRATION: Complete Monitoring Stack for Network AI API")
print("=" * 65)

random.seed(123)

# Initialize all monitoring components
api_client    = InstrumentedClient(team="NOC")
service_dash  = ServiceDashboard("network-ai-api", window_sec=300)
quality_mon   = QualityMonitor(faithfulness_threshold=0.80)
slo_tracker   = SLOTracker([
    SLO("availability", "success_rate",   target=0.99),
    SLO("speed",        "p95_latency_ms", target=4000.0, comparison="<="),
])
anomaly_det   = CostAnomalyDetector(window_size=10, threshold_sigma=2.5)

# Simulate a production monitoring cycle
scenarios = [
    # (query, context_snippet, expected_response, inject_error, inject_slow)
    ("What BGP attributes determine path selection?",
     BGP_CONTEXT,
     "BGP selects best path using LOCAL_PREF, AS_PATH length, ORIGIN, MED, eBGP over iBGP, IGP metric, Router-ID.",
     False, False),
    ("Explain OSPF DR election",
     "OSPF elects DR by highest priority, then Router-ID on broadcast/NBMA segments.",
     "OSPF DR election uses highest interface priority (default 1). If tied, highest Router-ID wins. DR never preempts.",
     False, False),
    ("BGP path selection uses OSPF metrics primarily",
     BGP_CONTEXT,
     "BGP path selection primarily uses OSPF routing metrics and link costs to determine best paths.",
     False, False),   # bad/hallucinated response
    ("What is QoS DSCP for voice?",
     "DSCP EF (46) is used for voice. AF41 for video. CS3 for signaling.",
     "QoS uses DSCP EF (46) for voice traffic. AF41 is for interactive video. CS3 is for call signaling.",
     False, False),
    ("Timeout query",
     "", "", True, True),   # simulated error
]

print()
print(f"{'#':<3} {'Query (40ch)':<42} {'Latency':>9} {'Cost':>9} {'Quality':>8} {'Status'}")
print("-" * 80)

for i, (query, ctx, response, inject_err, inject_slow) in enumerate(scenarios, 1):
    # Simulate different conditions
    if inject_err:
        latency = random.uniform(8000, 12000)
        is_success = False
        cost = 0.0
        quality_score = None
    elif inject_slow:
        latency = random.uniform(4000, 7000)
        is_success = True
        cost = random.uniform(0.001, 0.003)
        quality_score = None
    else:
        latency = random.uniform(200, 1200)
        is_success = True
        cost = random.uniform(0.0002, 0.001)
        # Evaluate quality for non-error calls
        if ctx and response:
            q_result = quality_mon.evaluate(query, ctx, response, source_id=f"call-{i}")
            quality_score = q_result.get("score", 0)
        else:
            quality_score = None

    # Record in all monitoring systems
    service_dash.record_call(latency, tokens_in=len(query.split())*3,
                             cost_usd=cost, is_error=not is_success)
    slo_tracker.record(is_success, latency_ms=latency, cost_usd=cost)
    anomaly_result = anomaly_det.record(cost, context=query[:20])

    status = "OK" if is_success else "ERROR"
    if anomaly_result.get("is_anomaly"):
        status = "COST-ANOMALY"

    q_str = f"{quality_score}/10" if quality_score else "-"
    print(f"{i:<3} {query[:40]:<42} {latency:>8.0f}ms ${cost:>8.5f} "
          f"{q_str:>8} {status}")

# ── Final status dashboard ────────────────────────────────────────────────────
print()
print("=== MONITORING DASHBOARD SNAPSHOT ===")
print()

# Service health
snap = service_dash.snapshot()
print("Service Health (RED):")
print(f"  p50: {snap['p50_ms']:.0f}ms | p95: {snap['p95_ms']:.0f}ms | "
      f"p99: {snap['p99_ms']:.0f}ms | errors: {snap['error_rate_pct']:.1f}%")

# SLO status
print()
print("SLO Status:")
slo_report = slo_tracker.status_report()
for name, status in slo_report["slo_status"].items():
    icon = "PASS" if status["passing"] else "BREACH"
    alert = " !! PAGE ON-CALL" if status["alert"] else ""
    print(f"  [{icon}] {name}: burn_rate={status['burn_rate']:.2f}{alert}")

# Quality summary
print()
print("Quality Monitor:")
q_sum = quality_mon.summary()
if q_sum:
    print(f"  Evaluations:       {q_sum['evaluations']}")
    print(f"  Avg faithfulness:  {q_sum['avg_faithfulness']:.3f}")
    print(f"  Avg relevance:     {q_sum['avg_relevance']:.3f}")
    print(f"  Alerts fired:      {q_sum['alerts_fired']}")
    if q_sum['alerts_fired'] > 0:
        print(f"  !! Quality degradation detected - check recent responses")

print()
print("Monitoring stack operational. In production:")
print("  - Ship structured logs -> Elasticsearch/Splunk")
print("  - Push metrics -> Prometheus -> Grafana dashboards")
print("  - SLO alerts -> PagerDuty/OpsGenie -> on-call engineer")
print("  - Quality alerts -> Slack channel -> NOC review queue")